# ???

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
%matplotlib inline

# ?????

In [ ]:
# ?????????
train_df = pd.read_csv('./data/train.csv')
test_df = pd.read_csv('./data/test.csv')

# ????

In [ ]:
# plt.scatter(train_df['????'], train_df['???'], c = "blue", marker = "s")
# plt.title("Looking for outliers")
# plt.xlabel("RoomArea")
# plt.ylabel("Rental")
# plt.show()

In [ ]:
# ?????,?1.94576???1.90965
train_df = train_df.drop(train_df[(train_df['????']>0.1)].index)

In [ ]:
# plt.scatter(train_df['????'], train_df['???'], c = "blue", marker = "s")
# plt.title("Looking for outliers")
# plt.xlabel("RoomArea")
# plt.ylabel("Rental")
# plt.show()

## ?????????

In [ ]:
drop_cols=['????????','??','????'] # ??????'????????','????','??'
# ???????labelEncoder
try:
    lb_encoder=LabelEncoder()
    lb_encoder.fit(train_df.loc[:,'????'].append(test_df.loc[:,'????'])) 
    train_df.loc[:,'????']=lb_encoder.transform(train_df.loc[:,'????'])
    test_df.loc[:,'????']=lb_encoder.transform(test_df.loc[:,'????'])
except Exception as e:
    print(e)

# ????????????????????
#     train_df.loc[:,'????']=train_df.loc[:,'??']/(train_df.loc[:,'???']+1)
#     test_df.loc[:,'????']=test_df.loc[:,'??']/(test_df.loc[:,'???']+1)
#     train_df.loc[:,'????']=train_df.loc[:,'????']+train_df.loc[:,'????']+train_df.loc[:,'????']
#     test_df.loc[:,'????']=test_df.loc[:,'????']+test_df.loc[:,'????']+test_df.loc[:,'????']
#     train_df.loc[:,'Area/Room']=train_df.loc[:,'????']/(train_df.loc[:,'????']+1)
#     test_df.loc[:,'Area/Room']=test_df.loc[:,'????']/(test_df.loc[:,'????']+1)
train_df['????']=train_df['??'] / (train_df['???'] + 1)
test_df['????']=test_df['??'] / (test_df['???'] + 1)
train_df['????']=train_df['????'] + train_df['????'] + train_df['????']
test_df['????']=test_df['????'] + test_df['????'] + test_df['????']
train_df['Area/Room']=train_df['????'] / (train_df['????']+1)
test_df['Area/Room']=test_df['????'] / (test_df['????']+1)

# ??????????????????rank????label
rank_cols=['??','????']
for col in rank_cols:
    # ?df?????col???????,????????,???'Rental'???????????,
    rank_df=train_df.loc[:,[col,'???']].groupby(col,as_index=False).mean().sort_values(by='???').reset_index(drop=True)
    rank_df.loc[:,col+'_rank']=rank_df.index+1        # +1,????????0??rank
    rank_fe_df=rank_df.drop(['???'],axis=1)
    # ?rank_fe_df??col????train_df&test_df
    train_df=train_df.merge(rank_fe_df,how='left',on=col)
    test_df=test_df.merge(rank_fe_df,how='left',on=col)
    # ?????col?
    try:
        train_df.drop([col],axis=1,inplace=True)
        test_df.drop([col],axis=1,inplace=True)
    except Exception as e:
        print(e)

# ??drop_cols?????
for drop_col in drop_cols:
        try:
            train_df.drop(drop_col,axis=1,inplace=True)
            test_df.drop(drop_col,axis=1,inplace=True)
        except Exception as e:
            pass

######################################################
## 1.904/0.754408??????
######################################################
#     # ??????????
#     train_df['BedLivingroom']=train_df['????']+train_df['????']
#     test_df['BedLivingroom']=test_df['????']+test_df['????']
#     train_df['DedBathroom']=train_df['????']+train_df['????']
#     test_df['DedBathroom']=test_df['????']+test_df['????']
#     train_df['LivBathroom']=train_df['????']+train_df['????']
#     test_df['LivBathroom']=test_df['????']+test_df['????']
# ????????????
#     train_df['BedroomRitio']=train_df['????']/train_df['????']
#     test_df['BedroomRitio']=test_df['????']/test_df['????']
#     train_df['LivingRitio']=train_df['????']/train_df['????']
#     test_df['LivingRitio']=test_df['????']/test_df['????']   
#     train_df['BathRitio']=train_df['????']/train_df['????']
#     test_df['BathRitio']=test_df['????']/test_df['????']    
#?????????????? 
train_df['????']=train_df['????']*(train_df['????']/train_df['????'])
test_df['????']=test_df['????']*(test_df['????']/test_df['????'])  
train_df['????']=train_df['????']*(train_df['????']/train_df['????'])
test_df['????']=test_df['????']*(test_df['????']/test_df['????'])   
train_df['????']=train_df['????']*(train_df['????']/train_df['????'])
test_df['????']=test_df['????']*(test_df['????']/test_df['????'])
#     train_df['else']=train_df['????'] - train_df['????']
#     test_df['else']=test_df['????'] - test_df['????']
######################################################
## 1.906/0.738673??????
######################################################
#     train_df['??'] = train_df['??'].fillna(1.)
#     test_df['??'] = test_df['??'].fillna(1.)
#     def fill_na(df):
#         a=df['????_rank'].dropna().mode()
#         df['????_rank'] = df['????_rank'].fillna(a)
#         a=df['????'].dropna().mode()
#         df['????'] = df['????'].fillna(a)
#         return df
#     train_df = train_df.groupby('???').apply(lambda x:fill_na(x))
#     test_df = test_df.groupby('???').apply(lambda x:fill_na(x))
#     # ??????????????????
#     items = ['????','????','????','????','???']
#     for item in items:
#         temp_train = train_df.groupby('???', as_index=False)[item].agg({item+'sum1':'sum'})
#         temp_test = test_df.groupby('???', as_index=False)[item].agg({item+'sum1':'sum'})
#     train_df.merge(temp_train, on='???', how='left')
#     test_df.merge(temp_test, on='???', how='left')

#     items = ['????','????','????','????','???']
#     for item in items:
#         temp_train = train_df.groupby('?', as_index=False)[item].agg({item+'sum2':'sum'})
#         temp_test = test_df.groupby('?', as_index=False)[item].agg({item+'sum2':'sum'})
#     train_df.merge(temp_train, on='?', how='left')
#     test_df.merge(temp_test, on='?', how='left')

# ???????????(????)
temp = train_df.groupby('???')['????'].count().reset_index()
temp.columns = ['???','??????']
train_df = train_df.merge(temp, how = 'left',on = '???')
test_df = test_df.merge(temp, how = 'left',on = '???')

temp1 = train_df.groupby('???')['????_rank'].count().reset_index()
temp1.columns = ['???','??????']
train_df = train_df.merge(temp1, how = 'left',on = '???') 
test_df = test_df.merge(temp1, how = 'left',on = '???')
######################################################
## 1.87246/0.72992??????
######################################################
train_df['PerFloorBedroomArea'] = train_df['????'] * train_df['????']
test_df['PerFloorBedroomArea'] = test_df['????'] * test_df['????']
#     train_df['PerFloorArea'] = train_df['????'] * train_df['????']
#     test_df['PerFloorArea'] = test_df['????'] * test_df['????']

# ????????
X_train=train_df.drop(['???'],axis=1)
Y_train=train_df.loc[:,'???'].values
test_id=test_df.loc[:,'id']
X_test=test_df.drop(['id'],axis=1)

reg_model=xgb.XGBRegressor(max_depth=8, # ??????,????????
                           n_estimators=3880, # ??????
                           learning_rate=0.1, # ???
                           n_jobs=-1) # ??cpu???
reg_model.fit(X_train, # ???
              Y_train, # ?????
              eval_set=[(X_train,Y_train)], # ???
              verbose = 10, # ???????????
              early_stopping_rounds=10 # ???10?rmse????,??????
             )

y_pred=reg_model.predict(X_test)

sub_df=pd.DataFrame({
    'id':test_id,
    'price':y_pred
})
sub_df.to_csv('./result/xgb.csv',index=False)
